
### MOUNT GOOGLE DRIVE

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### KONFIGURASI

In [2]:
CONFIG = {
    "dataset_path": "/content/drive/MyDrive/PDP_OCR_2/dataset",
    "result_path": "/content/drive/MyDrive/PDP_OCR_2/exp5_paddleV6/result_v6",
    "supported_ext": (".jpg", ".jpeg", ".png"),
    "start_index": 0,
    "limit": None,
    "gt_only": True,
    "gt_path": "/content/drive/MyDrive/PDP_OCR_2/groundtruth",
    "gt_extension": ".md",
    "lang": "en",
    "device": "gpu:0",
    "use_doc_orientation_classify": False,
    "use_doc_unwarping": False,
    "use_textline_orientation": True,
    "text_det_thresh": 0.3,
    "text_det_box_thresh": 0.6,
    "text_det_unclip_ratio": 1.5,
    "text_rec_score_thresh": 0.0,
}


import os
os.makedirs(CONFIG["result_path"], exist_ok=True)


### Instalasi PaddleOCR

In [3]:
!nvidia-smi --query-gpu=name,driver_version --format=csv,noheader
!python -c "import torch; print('CUDA runtime:', torch.version.cuda)" 2>/dev/null
!pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q "paddleocr>=3.0.0"
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --upgrade

Tesla T4, 580.82.07
CUDA runtime: 12.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.5.1+cu121 requires nvidia-cublas-cu12==12.1.3.1; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.6.4.1 which is incompatible.
torch 2.5.1+cu121 requires nvidia-cuda-cupti-cu12==12.1.105; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.6.80 which is incompatible.
torch 2.5.1+cu121 requires nvidia-cuda-nvrtc-cu12==12.1.105; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-nvrtc-cu12 12.6.77 which is incompatible.
torch 2.5.1+cu121 requires nvidia-cuda-runtime-cu12==12.1.105; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-runtime-cu12 12.6.77 which is incompatible.
torch 2.5.1+cu121 requires nvidia

### IMPORT LIBRARY

In [4]:
import os
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

import time
import numpy as np
from matplotlib import pyplot as plt
from PIL import Image, ImageDraw

if 'PaddleOCR' not in globals():
    from paddleocr import PaddleOCR

### INISIALISASI MODEL

In [5]:
if 'ocr' not in globals():
    try:
        ocr = PaddleOCR(
            lang=CONFIG["lang"],
            device=CONFIG["device"],
            use_doc_orientation_classify=CONFIG["use_doc_orientation_classify"],
            use_doc_unwarping=CONFIG["use_doc_unwarping"],
            use_textline_orientation=CONFIG["use_textline_orientation"],
            text_det_thresh=CONFIG["text_det_thresh"],
            text_det_box_thresh=CONFIG["text_det_box_thresh"],
            text_det_unclip_ratio=CONFIG["text_det_unclip_ratio"],
            text_rec_score_thresh=CONFIG["text_rec_score_thresh"],
        )
    except Exception as e:
        print(f"GPU gagal ({e}), fallback ke CPU")
        ocr = PaddleOCR(
            lang=CONFIG["lang"],
            device="cpu",
            use_doc_orientation_classify=CONFIG["use_doc_orientation_classify"],
            use_doc_unwarping=CONFIG["use_doc_unwarping"],
            use_textline_orientation=CONFIG["use_textline_orientation"],
            text_det_thresh=CONFIG["text_det_thresh"],
            text_det_box_thresh=CONFIG["text_det_box_thresh"],
            text_det_unclip_ratio=CONFIG["text_det_unclip_ratio"],
            text_rec_score_thresh=CONFIG["text_rec_score_thresh"],
        )



Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.


### HELPER FUNCTION


In [6]:
def draw_boxes(image, boxes, texts=None, scores=None):
    img = image.copy()
    draw = ImageDraw.Draw(img)
    for i, box in enumerate(boxes):
        pts = np.array(box).astype(np.int32).reshape(-1, 2)
        draw.polygon([tuple(p) for p in pts], outline="red", width=3)
        if texts is not None:
            label = texts[i] if scores is None else f"{texts[i]} ({scores[i]:.2f})"
            draw.text((pts[0][0], max(pts[0][1] - 15, 0)), label, fill="blue")
    return img


def show_image(img, title):
    plt.figure(figsize=(10, 6))
    plt.imshow(img)
    plt.title(title)
    plt.axis('off')
    plt.show()

def reconstruct_indentation(boxes, texts, tolerance=45, use_indent=True):
    if not boxes or not texts:
        return ""

    data = []
    for box, text in zip(boxes, texts):
        x_coords = [pt[0] for pt in box]
        y_coords = [pt[1] for pt in box]
        x_min = min(x_coords)
        y_center = sum(y_coords) / 4.0
        h = max(y_coords) - min(y_coords)
        data.append({"x": x_min, "y": y_center, "h": h, "text": text})

    data.sort(key=lambda item: item["y"])

    lines = []
    current_line = [data[0]]

    for item in data[1:]:
        prev_item = current_line[-1]
        if abs(item["y"] - prev_item["y"]) < max(item["h"], prev_item["h"]) * 0.5:
            current_line.append(item)
        else:
            lines.append(current_line)
            current_line = [item]
    lines.append(current_line)

    first_x_list = sorted([line[0]["x"] for line in lines])

    clusters = []
    if first_x_list:
        current_cluster = [first_x_list[0]]
        for x in first_x_list[1:]:
            if x - current_cluster[0] <= tolerance:
                current_cluster.append(x)
            else:
                clusters.append(current_cluster)
                current_cluster = [x]
        clusters.append(current_cluster)

    def get_indent_level(x_val):
        for i, cluster in enumerate(clusters):
            if x_val in cluster:
                return i
        return 0

    result_lines = []
    for line in lines:
        line.sort(key=lambda item: item["x"])
        first_x = line[0]["x"]

        # JIKA use_indent=True hitung levelnya, JIKA False langsung set ke 0 (rata kiri)
        level = get_indent_level(first_x) if use_indent else 0

        line_str = "    " * level + line[0]["text"]

        for i in range(1, len(line)):
            line_str += " " + line[i]["text"]

        result_lines.append(line_str)

    return "\n".join(result_lines)

def write_raw_markdown(image_name, text_content):
    md_path = os.path.join(CONFIG["result_path"], os.path.splitext(image_name)[0] + ".md")
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(text_content)


### MAIN FUNCTION

In [7]:
def process_ocr(image_name, index, total):
    img_path = os.path.join(CONFIG["dataset_path"], image_name)

    start = time.perf_counter()
    result = list(ocr.predict(img_path))
    elapsed = time.perf_counter() - start

    print(f"\n[{index}/{total}] {image_name} — {elapsed:.2f}s")
    print("=" * 50)

    if not result or len(result[0].get('rec_polys', [])) == 0:
        print("Tidak ada teks terdeteksi")
        return

    res = result[0]
    boxes = res['rec_polys']
    texts = res['rec_texts']
    scores = res['rec_scores']

    image = Image.open(img_path).convert('RGB')

    # STEP 1: Bounding Box
    box_only = draw_boxes(image, boxes)
    print(f"Step 1: Deteksi bounding box selesai ({len(boxes)} box)")
    show_image(box_only, f"{image_name} — Step 1: Bounding Box")

    # STEP 2: Hasil Recognition (Raw Text)
    print("\nStep 2: Hasil Recognition (Raw Text)")
    print("-" * 30)
    for i, t in enumerate(texts, 1):
        print(f"{i}. {t}")

    # STEP 3: Post-Processing Indentasi + Pengecualian
    base_name = os.path.splitext(image_name)[0]
    # Cek apakah nama file berakhiran '4_jawaban' atau 'penjelasan'
    if base_name.endswith("4_jawaban") or base_name.endswith("penjelasan"):
        use_indent = False
        print("\n[INFO] File terdeteksi narasi/penjelasan. Indentasi dinonaktifkan (Rata Kiri).")
    else:
        use_indent = True

    print("\nStep 3: Hasil Rekonstruksi Teks")
    print("-" * 30)
    reconstructed_text = reconstruct_indentation(boxes, texts, tolerance=45, use_indent=use_indent)
    print(reconstructed_text)
    print("=" * 50)

    # STEP 4: Simpan file markdown mentahan
    write_raw_markdown(image_name, reconstructed_text)
    print(f"File mentahan tersimpan: {base_name}.md")

###  EKSEKUSI PIPELINE

In [8]:
def has_gt(filename):
    base = os.path.splitext(filename)[0]
    return os.path.exists(os.path.join(CONFIG["gt_path"], base + CONFIG["gt_extension"]))


if os.path.exists(CONFIG["dataset_path"]):
    files = sorted(f for f in os.listdir(CONFIG["dataset_path"]) if f.lower().endswith(CONFIG["supported_ext"]))

    if CONFIG["gt_only"]:
        files = [f for f in files if has_gt(f)]

    start = CONFIG["start_index"]
    end = start + CONFIG["limit"] if CONFIG["limit"] else None
    files = files[start:end]

    total = len(files)
    print(f"Total foto yang akan diproses: {total}")
    print(f"Rentang index: {start} - {start + total - 1}")
    print(f"GT only: {CONFIG['gt_only']}")
    print("Daftar file:")
    for i, f in enumerate(files, start + 1):
        print(f"  [{i}] {f}")
    print("-" * 40)

    for idx, file in enumerate(files, 1):
        try:
            process_ocr(file, idx, total)
        except Exception as e:
            print(f"Skip {file}: {e}")
else:
    print(f"Dataset path not found: {CONFIG['dataset_path']}")

Output hidden; open in https://colab.research.google.com to view.

### BENCHMARK TEST

In [9]:
import os
import re

def levenshtein_dist(seq1, seq2):
    if len(seq1) < len(seq2):
        return levenshtein_dist(seq2, seq1)
    if len(seq2) == 0:
        return len(seq1)

    prev_row = range(len(seq2) + 1)
    for i, c1 in enumerate(seq1):
        curr_row = [i + 1]
        for j, c2 in enumerate(seq2):
            ins = prev_row[j + 1] + 1
            it_del = curr_row[j] + 1
            sub = prev_row[j] + (c1 != c2)
            curr_row.append(min(ins, it_del, sub))
        prev_row = curr_row
    return prev_row[-1]

def clean_text_norm(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def calculate_metrics(gt_text, ocr_text):
    # --- STANDARD (Raw) ---
    gt_std = gt_text.strip()
    ocr_std = ocr_text.strip()

    cer_std = levenshtein_dist(gt_std, ocr_std) / max(1, len(gt_std))
    wer_std = levenshtein_dist(gt_std.split(), ocr_std.split()) / max(1, len(gt_std.split()))

    # --- NORMALIZED ---
    gt_norm = clean_text_norm(gt_text)
    ocr_norm = clean_text_norm(ocr_text)

    cer_norm = levenshtein_dist(gt_norm, ocr_norm) / max(1, len(gt_norm))
    wer_norm = levenshtein_dist(gt_norm.split(), ocr_norm.split()) / max(1, len(gt_norm.split()))

    # --- KEYWORD ACCURACY (KW Acc) ---
    gt_words = set([w for w in gt_norm.split() if len(w) > 2])
    if gt_words:
        ocr_words = set(ocr_norm.split())
        matched_words = gt_words.intersection(ocr_words)
        kw_acc = len(matched_words) / len(gt_words)
    else:
        kw_acc = 1.0

    return cer_std, cer_norm, wer_std, wer_norm, kw_acc

# Jalankan evaluasi jika folder GT ada
if os.path.exists(CONFIG["gt_path"]) and os.path.exists(CONFIG["result_path"]):
    md_files = [f for f in os.listdir(CONFIG["result_path"]) if f.endswith(".md")]
    md_files = sorted(md_files)

    print("| File Name | CER Std | CER Norm | WER Std | WER Norm | KW Acc |")
    print("|---|---|---|---|---|---|")

    sum_cer_std, sum_cer_norm, sum_wer_std, sum_wer_norm, sum_kw_acc = 0, 0, 0, 0, 0
    valid_count = 0

    for md_file in md_files:
        base_name = os.path.splitext(md_file)[0]
        gt_file = base_name + CONFIG["gt_extension"]
        gt_full_path = os.path.join(CONFIG["gt_path"], gt_file)
        md_full_path = os.path.join(CONFIG["result_path"], md_file)

        if os.path.exists(gt_full_path):
            with open(gt_full_path, "r", encoding="utf-8") as f:
                gt_content = f.read()
            with open(md_full_path, "r", encoding="utf-8") as f:
                ocr_content = f.read()

            c_std, c_norm, w_std, w_norm, kw = calculate_metrics(gt_content, ocr_content)

            print(f"| {md_file} | {c_std*100:.2f}% | {c_norm*100:.2f}% | {w_std*100:.2f}% | {w_norm*100:.2f}% | {kw*100:.1f}% |")

            sum_cer_std += c_std
            sum_cer_norm += c_norm
            sum_wer_std += w_std
            sum_wer_norm += w_norm
            sum_kw_acc += kw
            valid_count += 1

    if valid_count > 0:
        avg_cer_std = (sum_cer_std / valid_count) * 100
        avg_cer_norm = (sum_cer_norm / valid_count) * 100
        avg_wer_std = (sum_wer_std / valid_count) * 100
        avg_wer_norm = (sum_wer_norm / valid_count) * 100
        avg_kw_acc = (sum_kw_acc / valid_count) * 100

        print("|---|---|---|---|---|---|")
        print(
            f"| **AVERAGE ({valid_count} Files)** | "
            f"**{avg_cer_std:.2f}%** | "
            f"**{avg_cer_norm:.2f}%** | "
            f"**{avg_wer_std:.2f}%** | "
            f"**{avg_wer_norm:.2f}%** | "
            f"**{avg_kw_acc:.1f}%** |"
        )
    else:
        print("Tidak ditemukan file Ground Truth (.md) yang cocok di folder target.")

| File Name | CER Std | CER Norm | WER Std | WER Norm | KW Acc |
|---|---|---|---|---|---|
| if4901_103011500026_1a_jawaban.md | 25.47% | 22.59% | 35.29% | 28.21% | 78.9% |
| if4901_103011500026_1b_jawaban.md | 35.78% | 26.19% | 50.68% | 50.91% | 56.7% |
| if4901_103011500026_2a_jawaban.md | 4.43% | 0.75% | 19.05% | 10.91% | 94.8% |
| if4901_103011500026_2b_jawaban.md | 29.27% | 30.43% | 33.96% | 33.68% | 78.3% |
| if4901_103011500026_3_jawaban.md | 13.59% | 12.83% | 26.09% | 25.00% | 85.2% |
| if4901_103011500026_4_jawaban.md | 67.81% | 63.13% | 80.95% | 70.92% | 11.8% |
| if4901_103011500114_1a_jawaban.md | 19.43% | 10.32% | 58.70% | 50.00% | 75.0% |
| if4901_103011500114_1b_jawaban.md | 37.15% | 22.30% | 80.00% | 74.32% | 56.1% |
| if4901_103011500114_2a_jawaban.md | 6.48% | 1.35% | 21.05% | 14.41% | 90.3% |
| if4901_103011500114_2b_jawaban.md | 7.00% | 1.57% | 15.32% | 12.96% | 91.0% |
| if4901_103011500114_3_jawaban.md | 22.93% | 8.11% | 24.14% | 17.22% | 100.0% |
| if4901_1030115